## Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
     

## Load data

In [0]:

df = spark.table("workspace.bronze.erp_cust_az12")

## Transform data

In [0]:
# trim

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
     

In [0]:
# id cleanup
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)


In [0]:
# birthday validation

df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

In [0]:
#gender normalization
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

In [0]:
#rename
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:

df.limit(10).display()
     

## Writing to db

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

In [0]:


%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10